# Module A: Coding Exercises

Fill in the blanks. Every cell contains scaffolded code with `___` placeholders. Replace them, run the cell, and the assert at the bottom will tell you if your answer is correct.

**How to use:**
1. Run the **Setup** cell first (loads libraries, generates synthetic data).
2. Work through Lessons 1, 2, 3 in order. Each has 10 exercises.
3. If you get stuck, expand the hint in the markdown above each exercise.
4. After the assert passes, move on. Do not skip; later exercises depend on earlier ones.

**Synthetic data is provided.** No internet, no API keys. The generators mimic SPY, AGG, GLD, BTC stylised behaviour.

## Setup: imports and synthetic data

Run this cell once. It defines:
- `prices_dict`: a dict mapping asset name to price series.
- `returns_dict`: a dict mapping asset name to log-return series.
- `Sigma_true`, `mu_true`: the ground-truth covariance and mean for portfolio exercises.
- `helper functions` for verification.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(2026)

ASSET_PARAMS = {
    'SPY': dict(mu=0.08, vol=0.15, df=5, leverage=-0.4),
    'AGG': dict(mu=0.04, vol=0.05, df=8, leverage=-0.1),
    'GLD': dict(mu=0.06, vol=0.18, df=6, leverage=0.0),
    'BTC': dict(mu=0.30, vol=0.80, df=3, leverage=-0.2),
}

def generate_prices(asset_name, n_years=15, seed=None):
    """Generate a price series with realistic stylised facts."""
    rng = np.random.default_rng(seed if seed is not None else abs(hash(asset_name)) % (2**31))
    p = ASSET_PARAMS[asset_name]
    n = int(n_years * 252)
    eps = rng.standard_t(p['df'], n)
    eps = eps / np.std(eps)
    daily_vol = p['vol'] / np.sqrt(252)
    daily_mu = p['mu'] / 252
    vol = np.zeros(n); vol[0] = daily_vol
    ret = np.zeros(n); ret[0] = daily_mu + vol[0] * eps[0]
    for t in range(1, n):
        shock = vol[t-1] * eps[t-1]
        asym = 1.0 + p['leverage'] * (1 if shock < 0 else 0)
        vol[t] = np.sqrt(1e-7 + 0.05 * (shock * asym)**2 + 0.93 * vol[t-1]**2)
        ret[t] = daily_mu + vol[t] * eps[t]
    prices = 100 * np.exp(np.cumsum(ret))
    idx = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=n)
    return pd.Series(prices, index=idx, name=asset_name)

prices_dict = {name: generate_prices(name, n_years=15) for name in ASSET_PARAMS}
returns_dict = {name: np.log(p / p.shift(1)).dropna() for name, p in prices_dict.items()}

# Ground truth for portfolio exercises (annualised)
mu_true = np.array([0.08, 0.04, 0.06, 0.10])
sigma_true = np.array([0.15, 0.05, 0.18, 0.20])
rho_true = np.array([
    [1.0,  -0.2, 0.1,  0.5],
    [-0.2,  1.0, 0.0,  -0.1],
    [0.1,   0.0, 1.0,  0.2],
    [0.5,  -0.1, 0.2,  1.0],
])
Sigma_true = np.outer(sigma_true, sigma_true) * rho_true

def approx_equal(a, b, tol=1e-4):
    return abs(float(a) - float(b)) < tol

print('Setup complete.')
print(f'  Assets: {list(prices_dict.keys())}')
print(f'  Each return series has {len(returns_dict["SPY"])} observations')
print(f'  Sigma_true shape: {Sigma_true.shape}')
print(f'  mu_true: {mu_true}')

---
# Lesson 1: Returns and Stylised Facts

Ten exercises building from price-to-return arithmetic up to a working GARCH simulator.

### Exercise 1.1: Simple and log returns

Compute simple and log returns from a price series. Drop the first row (NaN).

**Hint.** `prices.pct_change()` for simple. `np.log(prices / prices.shift(1))` for log.

In [ ]:
def simple_returns(prices: pd.Series) -> pd.Series:
    return ___

def log_returns(prices: pd.Series) -> pd.Series:
    return ___

# Sanity check
p = prices_dict['SPY']
rs = simple_returns(p)
rl = log_returns(p)
assert len(rs) == len(p) - 1, 'Wrong length: did you drop NaN?'
assert approx_equal(rs.iloc[0], p.iloc[1] / p.iloc[0] - 1)
assert approx_equal(rl.iloc[0], np.log(p.iloc[1] / p.iloc[0]))
print('Exercise 1.1 passed.')

### Exercise 1.2: Aggregation property of log returns

Verify that the sum of daily log returns equals the total log return $\ln(P_n / P_0)$. Then verify that the same does NOT hold for simple returns.

**Hint.** Compute `total_log = log(P_n / P_0)` and compare to `rl.sum()`.

In [ ]:
p = prices_dict['SPY']
rl = log_returns(p)
rs = simple_returns(p)

total_log_return = ___      # ln(P_last / P_first)
sum_of_log_returns = ___    # rl.sum()
total_simple_return = ___   # P_last / P_first - 1
sum_of_simple_returns = ___ # rs.sum()

assert approx_equal(total_log_return, sum_of_log_returns), 'Log returns must aggregate exactly'
# The next assert should pass: simple returns do NOT aggregate to total
assert not approx_equal(total_simple_return, sum_of_simple_returns), 'Simple returns should not aggregate'
print(f'Total log return:    {total_log_return:.4f}')
print(f'Sum of log returns:  {sum_of_log_returns:.4f}  (matches!)')
print(f'Total simple return: {total_simple_return:.4f}')
print(f'Sum of simples:      {sum_of_simple_returns:.4f}  (does NOT match)')
print('Exercise 1.2 passed.')

### Exercise 1.3: Annualisation

Compute annualised mean and volatility from daily log returns. Use 252 trading days per year.

**Hint.** Mean scales linearly: multiply by 252. Volatility scales with $\sqrt{252}$.

In [ ]:
def annualised_stats(daily_log_returns: pd.Series, periods_per_year: int = 252) -> dict:
    mean_ann = ___
    vol_ann = ___
    sharpe = ___  # mean_ann / vol_ann
    return dict(mean=mean_ann, vol=vol_ann, sharpe=sharpe)

rl = log_returns(prices_dict['SPY'])
stats_out = annualised_stats(rl)
assert 0.04 < stats_out['mean'] < 0.12, f"SPY mean should be roughly 8%; got {stats_out['mean']:.3f}"
assert 0.10 < stats_out['vol'] < 0.20, f"SPY vol should be roughly 15%; got {stats_out['vol']:.3f}"
print(f"Mean: {stats_out['mean']:.3%}, Vol: {stats_out['vol']:.3%}, Sharpe: {stats_out['sharpe']:.2f}")
print('Exercise 1.3 passed.')

### Exercise 1.4: Skewness and excess kurtosis

Compute the skewness and excess kurtosis of a return series. Excess kurtosis = kurtosis - 3.

**Hint.** `scipy.stats.skew` and `scipy.stats.kurtosis(r, fisher=True)` (Fisher convention is excess kurtosis directly).

In [ ]:
def higher_moments(returns: pd.Series) -> dict:
    skew = ___
    excess_kurt = ___  # use fisher=True so it returns kurt - 3 directly
    return dict(skew=skew, excess_kurt=excess_kurt)

for name in ['SPY', 'AGG', 'BTC']:
    m = higher_moments(returns_dict[name])
    print(f"{name}: skew={m['skew']:+.2f}, excess_kurt={m['excess_kurt']:+.2f}")

btc_kurt = higher_moments(returns_dict['BTC'])['excess_kurt']
agg_kurt = higher_moments(returns_dict['AGG'])['excess_kurt']
assert btc_kurt > agg_kurt, 'BTC should have heavier tails than AGG'
print('Exercise 1.4 passed.')

### Exercise 1.5: Fit a Student t and recover df

Use `scipy.stats.t.fit` to fit a Student t to the SPY returns. Recover the fitted df.

**Hint.** `df, loc, scale = stats.t.fit(returns)`. SPY synthetic data was generated with df = 5, so the recovered df should be in roughly the 4 to 7 range.

In [ ]:
def fit_student_t(returns: pd.Series) -> dict:
    df, loc, scale = ___
    return dict(df=df, loc=loc, scale=scale)

for name in ['SPY', 'AGG', 'BTC']:
    fit = fit_student_t(returns_dict[name])
    print(f"{name}: df={fit['df']:.2f}, loc={fit['loc']:+.5f}, scale={fit['scale']:.5f}")

spy_df = fit_student_t(returns_dict['SPY'])['df']
btc_df = fit_student_t(returns_dict['BTC'])['df']
assert btc_df < spy_df, 'BTC should fit a lower df (heavier tails) than SPY'
print('Exercise 1.5 passed.')

### Exercise 1.6: Compare normal vs t VaR estimates

Compute the 99% one-day VaR under (a) a fitted normal and (b) a fitted Student t. Compare to the empirical 99th percentile.

**Hint.** For losses (= negative returns), VaR_99 = -mu + sigma * z_{0.99}. Use `stats.norm.ppf(0.99)` and `stats.t.ppf(0.99, df)` for the quantiles.

In [ ]:
def var_estimates(returns: pd.Series, alpha: float = 0.99) -> dict:
    losses = -returns.values
    var_empirical = ___  # np.quantile of losses at alpha

    # Gaussian fit
    mu_n, sd_n = losses.mean(), losses.std(ddof=1)
    var_normal = ___  # mu_n + sd_n * stats.norm.ppf(alpha)

    # Student t fit
    df_t, loc_t, scale_t = stats.t.fit(losses)
    var_t = ___       # loc_t + scale_t * stats.t.ppf(alpha, df_t)

    return dict(empirical=var_empirical, normal=var_normal, t=var_t)

for name in ['SPY', 'BTC']:
    v = var_estimates(returns_dict[name], 0.99)
    print(f"{name}: empirical={v['empirical']:.2%}, normal={v['normal']:.2%}, t={v['t']:.2%}")

v = var_estimates(returns_dict['BTC'], 0.99)
assert v['t'] > v['normal'] * 0.95, 'For BTC, t-fit VaR should be similar to or larger than normal VaR'
print('Exercise 1.6 passed.')

### Exercise 1.7: Autocorrelation function

Compute the lag-1, lag-5, lag-22 autocorrelations of (a) returns and (b) absolute returns. Compare.

**Hint.** `pd.Series.autocorr(lag)` returns the autocorrelation at a given lag.

In [ ]:
def acf_at_lags(returns: pd.Series, lags=(1, 5, 22)) -> dict:
    abs_returns = ___  # take absolute value
    acf_r = {h: ___ for h in lags}        # autocorrelation of returns at lag h
    acf_abs = {h: ___ for h in lags}      # autocorrelation of |returns| at lag h
    return dict(returns=acf_r, abs_returns=acf_abs)

out = acf_at_lags(returns_dict['SPY'])
print(f"ACF of returns       at lags 1, 5, 22: {[f'{out[\"returns\"][h]:+.3f}' for h in (1,5,22)]}")
print(f"ACF of |returns|     at lags 1, 5, 22: {[f'{out[\"abs_returns\"][h]:+.3f}' for h in (1,5,22)]}")

# Volatility clustering: |returns| ACF should dwarf returns ACF
assert out['abs_returns'][1] > out['returns'][1], 'Volatility clustering: |returns| ACF should be larger than returns ACF at lag 1'
assert out['abs_returns'][5] > 0.05, 'SPY should show clustering at lag 5'
print('Exercise 1.7 passed.')

### Exercise 1.8: Ljung-Box test for autocorrelation

Use the Ljung-Box test to formally test for autocorrelation in returns and squared returns at lag 10.

**Hint.** `from statsmodels.stats.diagnostic import acorr_ljungbox`. Pass `lags=[10]`. The function returns a DataFrame with columns `lb_stat` and `lb_pvalue`.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

def ljung_box_diagnostic(returns: pd.Series, lag: int = 10) -> dict:
    lb_returns = ___       # acorr_ljungbox on returns, lags=[lag]
    lb_squared = ___       # acorr_ljungbox on returns**2, lags=[lag]
    return dict(
        p_returns=float(lb_returns['lb_pvalue'].iloc[0]),
        p_squared=float(lb_squared['lb_pvalue'].iloc[0]),
    )

out = ljung_box_diagnostic(returns_dict['SPY'], lag=10)
print(f"Ljung-Box p-value for returns:        {out['p_returns']:.4f}")
print(f"Ljung-Box p-value for squared returns: {out['p_squared']:.6f}")

# Squared returns should reject the null of zero autocorrelation
assert out['p_squared'] < 0.01, 'Squared returns should have highly significant autocorrelation'
print('Exercise 1.8 passed.')

### Exercise 1.9: Implement a GARCH(1,1) simulator

Build a GARCH(1,1) simulator: $\sigma_t^2 = \omega + \alpha r_{t-1}^2 + \beta \sigma_{t-1}^2$, with returns $r_t = \sigma_t z_t$ and $z_t \sim N(0, 1)$.

**Hint.** Initialize `vol[0]` to the long-run vol $\sqrt{\omega / (1 - \alpha - \beta)}$. Loop from t=1 to n-1.

In [ ]:
def simulate_garch(n: int, omega: float, alpha: float, beta: float, seed: int = 0) -> tuple:
    rng = np.random.default_rng(seed)
    z = rng.standard_normal(n)
    vol = np.zeros(n)
    ret = np.zeros(n)
    long_run_vol = ___  # sqrt(omega / (1 - alpha - beta))
    vol[0] = long_run_vol
    ret[0] = vol[0] * z[0]
    for t in range(1, n):
        vol[t] = ___  # sqrt(omega + alpha * ret[t-1]**2 + beta * vol[t-1]**2)
        ret[t] = ___  # vol[t] * z[t]
    return ret, vol

ret, vol = simulate_garch(n=2520, omega=1e-6, alpha=0.05, beta=0.93, seed=42)
assert len(ret) == 2520
# Volatility clustering: |ret| ACF at lag 1 should be substantial
abs_ret_lag1_acf = pd.Series(np.abs(ret)).autocorr(1)
assert abs_ret_lag1_acf > 0.10, f'GARCH should produce clustering, got |r| ACF = {abs_ret_lag1_acf:.3f}'
print(f'Realised |r| ACF at lag 1: {abs_ret_lag1_acf:.3f}')
print('Exercise 1.9 passed.')

### Exercise 1.10: The leverage effect

Compute the empirical correlation between today's return $r_t$ and tomorrow's realised vol $\sigma_{t+1}$. For SPY this should be negative (Black's leverage effect).

**Hint.** Use a 22-day rolling std as a proxy for $\sigma_{t+1}$. `returns.rolling(22).std()`. Then correlate `returns[:-1]` with `rolling_vol[1:]`. Drop NaNs.

In [ ]:
def leverage_effect(returns: pd.Series, vol_window: int = 22) -> float:
    rolling_vol = ___  # rolling std with window=vol_window
    r_today = returns.iloc[:-1].values
    vol_tomorrow = rolling_vol.shift(-1).iloc[:-1].values
    mask = ~np.isnan(r_today) & ~np.isnan(vol_tomorrow)
    return float(___)  # correlation between r_today[mask] and vol_tomorrow[mask]

for name in ['SPY', 'AGG', 'GLD']:
    c = leverage_effect(returns_dict[name])
    print(f'{name}: leverage correlation = {c:+.3f}')

spy_lev = leverage_effect(returns_dict['SPY'])
assert spy_lev < 0, f'SPY leverage effect should be negative, got {spy_lev:+.3f}'
print('Exercise 1.10 passed.')

---
# Lesson 2: Portfolio Variance from Scratch

Ten exercises building from `w' Sigma w` to the tangency portfolio.

### Exercise 2.1: Portfolio variance and mean

Implement the affine identity. `w` is a 1D array of weights, `Sigma` is a 2D covariance matrix, `mu` is a 1D mean vector.

**Hint.** `w @ Sigma @ w` for variance. `w @ mu` for mean.

In [ ]:
def portfolio_var(w: np.ndarray, Sigma: np.ndarray) -> float:
    return float(___)

def portfolio_mean(w: np.ndarray, mu: np.ndarray) -> float:
    return float(___)

w = np.array([0.25, 0.25, 0.25, 0.25])
v = portfolio_var(w, Sigma_true)
m = portfolio_mean(w, mu_true)
assert v > 0, 'Variance must be positive'
assert approx_equal(v, w @ Sigma_true @ w, tol=1e-8)
assert approx_equal(m, w @ mu_true, tol=1e-8)
print(f'Equal-weight portfolio: var={v:.5f}, vol={np.sqrt(v):.4f}, mean={m:.4f}')
print('Exercise 2.1 passed.')

### Exercise 2.2: Two-asset minimum variance closed form

Implement the closed-form minimum variance weight on asset 1 for two assets:
$$w_1^\star = \frac{\sigma_2^2 - \rho \sigma_1 \sigma_2}{\sigma_1^2 + \sigma_2^2 - 2 \rho \sigma_1 \sigma_2}$$

**Hint.** Watch the parentheses on the denominator.

In [ ]:
def min_var_two_asset(sigma1, sigma2, rho):
    num = ___  # sigma2**2 - rho * sigma1 * sigma2
    den = ___  # sigma1**2 + sigma2**2 - 2 * rho * sigma1 * sigma2
    return num / den

# When sigma1 == sigma2, optimal w1 should be 0.5 regardless of rho
for rho in [-0.9, -0.5, 0, 0.5, 0.9]:
    w1 = min_var_two_asset(0.20, 0.20, rho)
    assert approx_equal(w1, 0.5), f'Equal vols should give w1=0.5, got {w1} at rho={rho}'

# When sigma1 = 0.30 and sigma2 = 0.10, rho = 0, w1 should be small
w1 = min_var_two_asset(0.30, 0.10, 0)
assert approx_equal(w1, 0.10**2 / (0.30**2 + 0.10**2)), 'Closed form check failed'
print(f'sigma1=0.30, sigma2=0.10, rho=0: w1* = {w1:.4f}')
print('Exercise 2.2 passed.')

### Exercise 2.3: N-asset minimum variance

Implement the matrix formula $w_{\min} = \Sigma^{-1} \mathbf{1} / (\mathbf{1}^\top \Sigma^{-1} \mathbf{1})$.

**Hint.** `np.linalg.inv(Sigma)` and `np.ones(n)`. The denominator is a scalar.

In [ ]:
def min_var_portfolio(Sigma: np.ndarray) -> np.ndarray:
    n = Sigma.shape[0]
    inv = ___  # np.linalg.inv(Sigma)
    ones = ___ # np.ones(n)
    w_unnorm = inv @ ones
    return w_unnorm / w_unnorm.sum()

w = min_var_portfolio(Sigma_true)
assert approx_equal(w.sum(), 1.0), 'Weights must sum to 1'
min_var = portfolio_var(w, Sigma_true)
ew_var = portfolio_var(np.ones(4) / 4, Sigma_true)
assert min_var < ew_var, 'Min-var portfolio should beat equal-weight'
print(f'Min-var weights:    {w.round(3)}')
print(f'Min-var vol:        {np.sqrt(min_var):.4f}')
print(f'Equal-weight vol:   {np.sqrt(ew_var):.4f}')
print('Exercise 2.3 passed.')

### Exercise 2.4: Eigenvalues and condition number

Compute the eigenvalues of `Sigma_true`, and the condition number $\kappa = \lambda_{\max} / \lambda_{\min}$.

**Hint.** `np.linalg.eigvalsh` returns sorted eigenvalues for a symmetric matrix.

In [ ]:
def eigen_diagnostics(Sigma: np.ndarray) -> dict:
    eigvals = ___  # eigvalsh
    cond = ___     # max / min
    return dict(
        eigvals=eigvals,
        cond=cond,
        psd_ok=bool(eigvals.min() >= -1e-10),
    )

diag = eigen_diagnostics(Sigma_true)
print(f'Eigenvalues:      {diag["eigvals"].round(5)}')
print(f'Condition number: {diag["cond"]:.2f}')
print(f'PSD ok:           {diag["psd_ok"]}')
assert diag['psd_ok'], 'Sigma_true should be PSD'
assert diag['cond'] > 1, 'Condition number must be at least 1'
print('Exercise 2.4 passed.')

### Exercise 2.5: Empirical verification of the affine identity

Simulate from a multivariate normal with mean `mu_true` and covariance `Sigma_true`. Compute the portfolio's empirical variance and verify it matches `w' Sigma w`.

**Hint.** `np.random.default_rng(seed).multivariate_normal(mu, Sigma, size=N)`. The empirical variance from N=100,000 samples should match the theoretical value to about 1%.

In [ ]:
def verify_affine_identity(w, mu, Sigma, n_samples=100_000, seed=0):
    rng = np.random.default_rng(seed)
    samples = ___  # multivariate_normal samples of shape (n_samples, len(w))
    portfolio_returns = ___  # samples @ w
    empirical_var = ___      # variance of portfolio_returns
    theoretical_var = portfolio_var(w, Sigma)
    return dict(empirical=empirical_var, theoretical=theoretical_var)

w = np.array([0.4, 0.3, 0.2, 0.1])
out = verify_affine_identity(w, mu_true, Sigma_true)
print(f"Empirical variance:   {out['empirical']:.6f}")
print(f"Theoretical w'Sigma w: {out['theoretical']:.6f}")
assert abs(out['empirical'] - out['theoretical']) / out['theoretical'] < 0.05, 'Should match to 5%'
print('Exercise 2.5 passed.')

### Exercise 2.6: Diversification ratio

Implement the Choueifaty-Coignard diversification ratio:
$$DR(w) = \frac{w^\top \sigma}{\sqrt{w^\top \Sigma w}}$$
where $\sigma$ is the vector of individual asset volatilities (the diagonal of $\Sigma$ square-rooted).

**Hint.** `np.sqrt(np.diag(Sigma))` gives the vol vector.

In [ ]:
def diversification_ratio(w: np.ndarray, Sigma: np.ndarray) -> float:
    sigmas = ___  # individual asset volatilities
    weighted_avg_vol = ___  # w' sigma
    portfolio_vol = ___     # sqrt(w' Sigma w)
    return weighted_avg_vol / portfolio_vol

w_eq = np.ones(4) / 4
dr_eq = diversification_ratio(w_eq, Sigma_true)
dr_single = diversification_ratio(np.array([1, 0, 0, 0]), Sigma_true)
assert dr_eq > 1.0, 'Equal weight on diversified assets should give DR > 1'
assert approx_equal(dr_single, 1.0, tol=1e-6), 'Single asset has DR = 1 by construction'
print(f'Equal-weight DR: {dr_eq:.3f}')
print(f'Single-asset DR: {dr_single:.3f}')
print('Exercise 2.6 passed.')

### Exercise 2.7: Tangency portfolio (with risk-free rate)

Implement the tangency (max Sharpe) portfolio:
$$w_T = \frac{\Sigma^{-1} (\mu - r_f \mathbf{1})}{\mathbf{1}^\top \Sigma^{-1} (\mu - r_f \mathbf{1})}$$

**Hint.** Use `np.linalg.solve(Sigma, excess)` instead of inverting Sigma explicitly. It is more numerically stable.

In [ ]:
def tangency_portfolio(mu: np.ndarray, Sigma: np.ndarray, rf: float = 0.0) -> np.ndarray:
    excess = ___  # mu - rf
    w_unnorm = ___  # np.linalg.solve(Sigma, excess)
    return w_unnorm / w_unnorm.sum()

w_t = tangency_portfolio(mu_true, Sigma_true, rf=0.02)
sharpe = (portfolio_mean(w_t, mu_true) - 0.02) / np.sqrt(portfolio_var(w_t, Sigma_true))
print(f'Tangency weights: {w_t.round(3)}')
print(f'Tangency Sharpe:  {sharpe:.3f}')

# Equal-weight Sharpe
w_eq = np.ones(4) / 4
ew_sharpe = (portfolio_mean(w_eq, mu_true) - 0.02) / np.sqrt(portfolio_var(w_eq, Sigma_true))
assert sharpe > ew_sharpe, 'Tangency should beat equal-weight on Sharpe'
print('Exercise 2.7 passed.')

### Exercise 2.8: Numerical efficient frontier

Sweep target returns from min(mu) to max(mu) in 30 steps. For each, find the minimum variance portfolio achieving that return. Plot the frontier.

**Hint.** This is a constrained quadratic program. Use `scipy.optimize.minimize` with constraints: `w' mu == target`, `w.sum() == 1`.

In [ ]:
from scipy.optimize import minimize

def efficient_frontier(mu, Sigma, n_points=30):
    n = len(mu)
    targets = np.linspace(mu.min(), mu.max(), n_points)
    vols = []
    for tgt in targets:
        constraints = [
            {'type': 'eq', 'fun': lambda w: ___ },           # weights sum to 1
            {'type': 'eq', 'fun': lambda w, t=tgt: ___ },    # mean equals target
        ]
        result = minimize(
            fun=lambda w: ___,                                # minimise w' Sigma w
            x0=np.ones(n) / n,
            constraints=constraints,
            method='SLSQP',
        )
        vols.append(np.sqrt(result.fun))
    return np.array(targets), np.array(vols)

targets, vols = efficient_frontier(mu_true, Sigma_true)
plt.figure(figsize=(8, 5))
plt.plot(vols, targets, 'b-', linewidth=2, label='Efficient frontier')
plt.scatter(np.sqrt(np.diag(Sigma_true)), mu_true, c='red', s=80, label='Individual assets', zorder=5)
plt.xlabel('Portfolio volatility')
plt.ylabel('Portfolio expected return')
plt.title('Efficient frontier')
plt.legend(); plt.grid(alpha=0.3); plt.show()
assert vols.min() < np.diag(Sigma_true).min() ** 0.5, 'Min-var should be below any single asset vol'
print('Exercise 2.8 passed.')

### Exercise 2.9: Inverse-volatility weighting

Implement inverse-volatility weighting (a naive risk parity). $w_i \propto 1 / \sigma_i$.

**Hint.** Compute the inverse vols, normalise to sum to 1.

In [ ]:
def inverse_vol_portfolio(Sigma: np.ndarray) -> np.ndarray:
    sigmas = ___  # individual vols
    inv = ___     # 1 / sigmas
    return inv / inv.sum()

w_iv = inverse_vol_portfolio(Sigma_true)
print(f'Inverse-vol weights: {w_iv.round(3)}')
# Lowest-vol asset should get the largest weight
assert w_iv.argmax() == np.diag(Sigma_true).argmin(), 'Lowest vol asset should have largest weight'
print('Exercise 2.9 passed.')

### Exercise 2.10: Risk contributions

Compute each asset's risk contribution under a portfolio: $RC_i = w_i (\Sigma w)_i / \sqrt{w^\top \Sigma w}$. They should sum to portfolio vol.

**Hint.** This is the building block for Risk Parity (Lesson 14).

In [ ]:
def risk_contributions(w: np.ndarray, Sigma: np.ndarray) -> dict:
    sigma_p = ___  # sqrt(w' Sigma w)
    marginal = ___ # Sigma @ w / sigma_p
    rc = ___       # w * marginal
    rc_share = rc / rc.sum()
    return dict(sigma_p=sigma_p, rc=rc, rc_share=rc_share)

w = np.array([0.6, 0.4, 0.0, 0.0])
out = risk_contributions(w, Sigma_true)
assert approx_equal(out['rc'].sum(), out['sigma_p']), 'RC should sum to portfolio vol'
print(f"Portfolio vol: {out['sigma_p']:.4f}")
print(f"Risk contributions:    {out['rc'].round(4)}")
print(f"Risk contribution share: {(out['rc_share']*100).round(1)} %")
print('Exercise 2.10 passed.')

---
# Lesson 3: Risk Measures from Variance to Drawdown

Ten exercises building from VaR to coherent risk and drawdown.

### Exercise 3.1: Historical VaR and CVaR

Given a 1D array of losses (positive = loss), compute the $\alpha$-VaR (empirical quantile) and the CVaR (mean of losses at or above the VaR).

**Hint.** `np.quantile` for VaR. Boolean indexing `losses[losses >= var]` for the tail.

In [ ]:
def historical_var_cvar(losses, alpha=0.95):
    losses = np.asarray(losses)
    var = ___  # quantile at alpha
    tail = ___ # losses at or above var
    cvar = ___ # mean of tail
    return dict(var=float(var), cvar=float(cvar))

losses = -returns_dict['SPY'].values
out = historical_var_cvar(losses, alpha=0.99)
assert out['cvar'] >= out['var'], 'CVaR must be >= VaR'
print(f"99% VaR:  {out['var']:.4f}")
print(f"99% CVaR: {out['cvar']:.4f}")
print('Exercise 3.1 passed.')

### Exercise 3.2: Parametric Gaussian VaR and CVaR

Closed forms under the Gaussian assumption.

$\text{VaR}^N_\alpha = \mu + \sigma z_\alpha$

$\text{CVaR}^N_\alpha = \mu + \sigma \frac{\phi(z_\alpha)}{1 - \alpha}$

**Hint.** `stats.norm.ppf(alpha)` for $z_\alpha$, `stats.norm.pdf(z)` for $\phi$.

In [ ]:
def gaussian_var_cvar(losses, alpha=0.95):
    mu = losses.mean()
    sigma = losses.std(ddof=1)
    z = ___  # stats.norm.ppf(alpha)
    phi = ___ # stats.norm.pdf(z)
    var = ___ # mu + sigma * z
    cvar = ___ # mu + sigma * phi / (1 - alpha)
    return dict(var=float(var), cvar=float(cvar))

losses = -returns_dict['SPY'].values
g = gaussian_var_cvar(losses, alpha=0.99)
h = historical_var_cvar(losses, alpha=0.99)
print(f"Gaussian   VaR={g['var']:.4f}, CVaR={g['cvar']:.4f}")
print(f"Historical VaR={h['var']:.4f}, CVaR={h['cvar']:.4f}")
# At 99%, historical CVaR usually exceeds Gaussian CVaR for fat-tailed data
print(f'Ratio CVaR_hist / CVaR_norm: {h["cvar"]/g["cvar"]:.2f}')
print('Exercise 3.2 passed.')

### Exercise 3.3: Parametric Student t VaR

Same as 3.2 but using a fitted Student t.

**Hint.** Fit with `stats.t.fit(losses)`. Then $\text{VaR}^t_\alpha = \text{loc} + \text{scale} \cdot t_{df}^{-1}(\alpha)$.

In [ ]:
def t_var(losses, alpha=0.95):
    df, loc, scale = ___  # stats.t.fit
    q = ___               # stats.t.ppf(alpha, df)
    var = ___             # loc + scale * q
    return dict(var=float(var), df=float(df))

losses = -returns_dict['BTC'].values
t_out = t_var(losses, alpha=0.99)
g_out = gaussian_var_cvar(losses, alpha=0.99)
print(f"BTC fitted df: {t_out['df']:.2f}")
print(f"BTC t-VaR (99%):     {t_out['var']:.4f}")
print(f"BTC Gaussian VaR:    {g_out['var']:.4f}")
# Check df is in expected range for crypto
assert 2.0 < t_out['df'] < 6.0, f'BTC df should be heavy-tailed, got {t_out["df"]:.2f}'
print('Exercise 3.3 passed.')

### Exercise 3.4: Maximum drawdown

Compute the peak-to-trough max drawdown of an equity curve.

**Hint.** `np.maximum.accumulate(equity)` gives the running peak. Drawdown at $t$ = $1 - W_t / \text{peak}_t$.

In [ ]:
def max_drawdown(equity: pd.Series) -> dict:
    eq = equity.values
    peak = ___  # running max
    dd = ___    # 1 - eq / peak
    trough_i = int(___)  # argmax of dd
    peak_i = int(___)    # argmax of eq up to trough_i (inclusive)
    return dict(
        max_dd=float(dd[trough_i]),
        peak_date=equity.index[peak_i],
        trough_date=equity.index[trough_i],
    )

eq = (1 + returns_dict['SPY']).cumprod()
out = max_drawdown(eq)
assert 0 < out['max_dd'] < 1, 'Max drawdown should be a fraction in (0, 1)'
assert out['peak_date'] < out['trough_date'], 'Peak must come before trough'
print(f"Max drawdown:    {out['max_dd']:.2%}")
print(f"From: {out['peak_date'].date()}")
print(f"To:   {out['trough_date'].date()}")
print('Exercise 3.4 passed.')

### Exercise 3.5: Conditional drawdown at risk (CDaR)

Average drawdown across the worst $1 - \alpha$ fraction of the path.

**Hint.** Compute the drawdown series, take the empirical $\alpha$-quantile of dd, average dd values at or above that threshold.

In [ ]:
def cdar(equity: pd.Series, alpha: float = 0.95) -> float:
    eq = equity.values
    peak = np.maximum.accumulate(eq)
    dd = 1 - eq / peak
    threshold = ___  # quantile of dd at alpha
    tail = ___       # dd values at or above threshold
    return float(___)  # mean of tail

for name in ['SPY', 'AGG', 'BTC']:
    eq = (1 + returns_dict[name]).cumprod()
    c = cdar(eq, alpha=0.95)
    print(f'{name}: CDaR_95 = {c:.2%}')

btc_cdar = cdar((1 + returns_dict['BTC']).cumprod(), 0.95)
agg_cdar = cdar((1 + returns_dict['AGG']).cumprod(), 0.95)
assert btc_cdar > agg_cdar, 'BTC should have deeper drawdowns than AGG'
print('Exercise 3.5 passed.')

### Exercise 3.6: Sortino ratio

Like Sharpe but uses downside deviation instead of total volatility. Downside deviation = std of returns below a target (typically 0 or rf).

**Hint.** Mask returns below the target, square them, mean, sqrt.

In [ ]:
def sortino_ratio(returns: pd.Series, target: float = 0.0, periods_per_year: int = 252) -> float:
    excess = returns - target
    downside = ___  # excess values below 0 (or all squared if you prefer the technical definition)
    downside_dev = ___  # sqrt(mean of squared downside)
    if downside_dev == 0:
        return float('nan')
    annualised_excess = excess.mean() * periods_per_year
    annualised_dd = downside_dev * np.sqrt(periods_per_year)
    return annualised_excess / annualised_dd

for name in ['SPY', 'AGG', 'GLD']:
    s = sortino_ratio(returns_dict[name])
    print(f'{name} Sortino: {s:.2f}')
print('Exercise 3.6 passed.')

### Exercise 3.7: Calmar ratio

Annualised return divided by max drawdown.

**Hint.** Reuse `max_drawdown` from Exercise 3.4.

In [ ]:
def calmar_ratio(returns: pd.Series, periods_per_year: int = 252) -> float:
    eq = (1 + returns).cumprod()
    mdd = ___  # call max_drawdown
    ann_return = ___  # mean * periods_per_year
    if mdd == 0:
        return float('nan')
    return ann_return / mdd

for name in ['SPY', 'AGG', 'GLD', 'BTC']:
    c = calmar_ratio(returns_dict[name])
    print(f'{name} Calmar: {c:.2f}')
print('Exercise 3.7 passed.')

### Exercise 3.8: CVaR sub-additivity check

Empirically verify that for two real assets, $\text{CVaR}(X + Y) \le \text{CVaR}(X) + \text{CVaR}(Y)$.

**Hint.** Take SPY and AGG. Form a 50/50 portfolio of returns. Compute CVaR of each leg and the portfolio.

In [ ]:
def check_cvar_subadditivity(r1, r2, weight1=0.5, alpha=0.95):
    portfolio = ___  # weight1 * r1 + (1 - weight1) * r2
    cvar_1 = historical_var_cvar(-r1.values, alpha)['cvar']
    cvar_2 = historical_var_cvar(-r2.values, alpha)['cvar']
    cvar_p = historical_var_cvar(-portfolio.values, alpha)['cvar']
    bound = weight1 * cvar_1 + (1 - weight1) * cvar_2
    return dict(cvar_p=cvar_p, bound=bound, sub_additive=cvar_p <= bound + 1e-8)

# Align indices
spy = returns_dict['SPY']
agg = returns_dict['AGG'].reindex(spy.index).fillna(0)
out = check_cvar_subadditivity(spy, agg, weight1=0.5, alpha=0.95)
print(f"Portfolio CVaR:           {out['cvar_p']:.4f}")
print(f"0.5 CVaR(A) + 0.5 CVaR(B): {out['bound']:.4f}")
print(f"Sub-additive:              {out['sub_additive']}")
assert out['sub_additive']
print('Exercise 3.8 passed.')

### Exercise 3.9: Kupiec POF test for VaR

Given a daily VaR forecast, count exceedances (days where actual loss > VaR). Under the null, the count is Binomial(n, 1 - alpha). Compute the POF likelihood ratio statistic.

**Hint.** $LR = -2 \ln \big[ ((1-\alpha)^N \alpha^{n-N}) / ((N/n)^N (1 - N/n)^{n-N}) \big]$. Reject at 95% if $LR > 3.84$.

In [ ]:
def kupiec_pof_test(losses, var_forecast, alpha=0.95):
    n = len(losses)
    N = int(___)  # number of exceedances: losses > var_forecast
    p_hat = N / n
    if N == 0 or N == n:
        return dict(N=N, n=n, lr=0.0, reject=False)
    p_null = 1 - alpha
    log_l_null = N * np.log(p_null) + (n - N) * np.log(alpha)
    log_l_alt = N * np.log(p_hat) + (n - N) * np.log(1 - p_hat)
    lr = ___  # -2 * (log_l_null - log_l_alt)
    return dict(N=N, n=n, lr=float(lr), reject=bool(lr > 3.84))

# Use a constant VaR forecast (Gaussian) and check exceedances
losses = -returns_dict['SPY'].values
g_var = gaussian_var_cvar(losses, 0.99)['var']
out = kupiec_pof_test(losses, g_var, alpha=0.99)
print(f"n={out['n']}, exceedances N={out['N']}")
print(f"Expected ~{int(out['n'] * 0.01)} under null")
print(f"LR = {out['lr']:.2f}, reject Gaussian VaR? {out['reject']}")
print('Exercise 3.9 passed.')

### Exercise 3.10: Compare risk measures across asset classes

Build a summary table with annualised vol, 99% VaR, 99% CVaR, max drawdown, Sortino, Calmar for SPY, AGG, GLD, BTC.

**Hint.** Reuse functions from Exercises 1.3, 3.1, 3.4, 3.6, 3.7.

In [ ]:
def risk_summary(returns: pd.Series) -> dict:
    losses = -returns.values
    eq = (1 + returns).cumprod()
    vc = historical_var_cvar(losses, 0.99)
    return dict(
        vol_ann = ___,                    # annualised vol
        var_99 = vc['var'],
        cvar_99 = vc['cvar'],
        max_dd = ___,                     # max_drawdown(eq)['max_dd']
        sortino = ___,                    # sortino_ratio(returns)
        calmar = ___,                     # calmar_ratio(returns)
    )

summary = pd.DataFrame({name: risk_summary(returns_dict[name]) for name in ['SPY','AGG','GLD','BTC']}).T
print(summary.round(4))
assert summary.loc['BTC', 'vol_ann'] > summary.loc['AGG', 'vol_ann']
assert summary.loc['BTC', 'max_dd'] > summary.loc['AGG', 'max_dd']
print('Exercise 3.10 passed. Module A complete!')

---
## Bonus: integrative exercises

Three open-ended exercises that combine concepts across all three lessons.

### Bonus B.1: VaR backtest with rolling Gaussian forecasts

Build a daily 99% VaR forecast for SPY using a rolling 250-day Gaussian fit. Apply the Kupiec test to the resulting series.

**Hint.** Use `returns.rolling(250).std()` and `returns.rolling(250).mean()` to build a time-varying VaR. Compare with the next-day realised loss.

In [ ]:
def rolling_var_backtest(returns: pd.Series, window: int = 250, alpha: float = 0.99) -> dict:
    rolling_mean = ___  # rolling mean
    rolling_sd = ___    # rolling std
    z = stats.norm.ppf(alpha)
    var_forecast = -rolling_mean + rolling_sd * z   # loss level
    var_forecast = var_forecast.shift(1)            # use yesterday's forecast for today
    aligned = pd.concat([-returns, var_forecast], axis=1, keys=['loss', 'var']).dropna()
    return kupiec_pof_test(aligned['loss'].values, aligned['var'].values, alpha=alpha)

out = rolling_var_backtest(returns_dict['SPY'], window=250, alpha=0.99)
print(out)
print('Bonus B.1 passed.')

### Bonus B.2: Build a 4-asset minimum variance portfolio from real-ish data

Estimate Sigma from the synthetic returns (rather than using the ground truth). Compare your estimated min-var weights to the ground-truth min-var weights.

**Hint.** Stack the four return series into a DataFrame, take `cov()`, multiply by 252 to annualise.

In [ ]:
names = ['SPY', 'AGG', 'GLD', 'BTC']
rets_df = pd.concat([returns_dict[n] for n in names], axis=1, keys=names).dropna()
Sigma_est = ___  # rets_df.cov() * 252
Sigma_est = np.asarray(Sigma_est)

w_est = min_var_portfolio(Sigma_est)
w_true = min_var_portfolio(Sigma_true)
print(f'Estimated weights: {dict(zip(names, w_est.round(3)))}')
print(f'True weights:      {dict(zip(names, w_true.round(3)))}')
# Difference is the price of estimation error
diff = np.linalg.norm(w_est - w_true)
print(f'L2 weight difference: {diff:.3f}')
print('Bonus B.2 passed.')

### Bonus B.3: Tangency vs equal weight, out of sample

Split the data 70/30 (train/test). Estimate Sigma and mu on train. Build tangency, min-var, equal-weight, inverse-vol portfolios. Compute realised Sharpe on test. Tabulate.

**Hint.** This is a 5-line preview of Lessons 9 and 23. The tangency portfolio will likely lose to equal weight out of sample. That is the central observation that Module B (Markowitz) and Module C (estimation error) build on.

In [ ]:
names = ['SPY', 'AGG', 'GLD', 'BTC']
rets_df = pd.concat([returns_dict[n] for n in names], axis=1, keys=names).dropna()
split = int(len(rets_df) * 0.7)
train, test = rets_df.iloc[:split], rets_df.iloc[split:]

mu_hat = ___                # train.mean() * 252
Sigma_hat = ___             # train.cov() * 252
mu_hat = np.asarray(mu_hat); Sigma_hat = np.asarray(Sigma_hat)

ports = {
    'tangency':    tangency_portfolio(mu_hat, Sigma_hat, rf=0.02),
    'min_var':     min_var_portfolio(Sigma_hat),
    'equal_weight': np.ones(4) / 4,
    'inverse_vol': inverse_vol_portfolio(Sigma_hat),
}

rows = []
for label, w in ports.items():
    daily_pnl = test.values @ w
    sr = daily_pnl.mean() / daily_pnl.std(ddof=1) * np.sqrt(252)
    rows.append((label, w.round(3).tolist(), sr))
out_df = pd.DataFrame(rows, columns=['portfolio', 'weights', 'oos_sharpe']).set_index('portfolio')
print(out_df)
print('Bonus B.3 passed.')

---
## Wrap up

If you got this far, you have a working scaffold for everything Module A teaches: returns, stylised facts, the affine identity, min-var portfolios, tangency, and the four major risk measures (variance, VaR, CVaR, drawdown).

**Next step.** When you are ready to start Week 2 (Markowitz core), open `lesson_05_markowitz.html` (to be built) and the matching `module_b_exercises.ipynb`.

**Where the bodies are buried.** The bonus exercises hint at the central problem of the next module: estimation error. The tangency portfolio you computed in B.3 looks beautiful in sample but loses out of sample. That phenomenon is the reason Lessons 9 to 13 exist.